## Imports & Paths

In [20]:
from pathlib import Path

import matplotlib.pyplot as plt
import nir
import numpy as np
import torch
from torch.utils.data import TensorDataset

from InternalSimulator import NIR2FPGA
from InternalSimulator.DiscretizationChoices import DiscretizationChoices, PTQOptions
from InternalSimulator.HardwareSimulation import SimulationOptions
from InternalSimulator.PredictionType import OutputEqual

torch.manual_seed(42)

output_dir = Path("./outputs")
norse_dir  = output_dir / "norse"
output_dir.mkdir(parents=True, exist_ok=True)

primitives_dir = Path("/home/mrontio/uni/phd/documents/26iscas/primitive_implementations")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# 1. Load saved files

In [21]:
dataset     = torch.from_numpy(np.load(str(output_dir / "dataset.npy")))        # (10, 200, 1)
print(f"Dataset: {dataset.shape}")

Dataset: torch.Size([10, 200, 1])


In [22]:
nir_lif_norse  = nir.read(str(norse_dir  / "lif.nir"))

In [23]:
dc_dataset = TensorDataset(
    dataset,
    torch.zeros(dataset.shape[0], dtype=torch.long),  # no classification labels needed
)

dc = DiscretizationChoices(
    timesteps=dataset.shape[1], # 200
    dataset=dc_dataset,
    batch_size=dataset.shape[0], # 10
    total_bits=16,
    macWidth=4,
    representative_sample_index=42,
    ptq=PTQOptions(method="minmax"),
)

# 3. Norse

In [24]:
norse_source = torch.load(str(norse_dir / "lif_source.pt"), weights_only=True)
n2f_norse = NIR2FPGA("lif-norse", nir_lif_norse, dc,
                     simulation_options=SimulationOptions(
                         primitives_dir=str(primitives_dir)
                     ))
n2f_norse.add_recording("source", norse_source)
n2f_norse.report_quantization()
n2f_norse.save_files()
n2f_norse.simulate()

TypeError: SimulationOptions.__init__() got an unexpected keyword argument 'primitives_dir'

In [ ]:
n2f_norse.evaluate_characteristic(2, output_dir = norse_dir / "characteristics")

## Plot

In [11]:
n2f_norse.plot("0", 0,  filename=str(figures_dir / "n2f.png"), ignorelist=[], plot_size=(20, 10))